In [28]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

project_root

WindowsPath('c:/Users/Dell - i5 11th Gen/Desktop/atm-protein-conservation-explorer')

In [29]:
# get protein representatives

protein_file = (
    project_root
    / "data"
    / "processed"
    / "atm_representative_proteins.csv"
)

protein_metadata = pd.read_csv(protein_file)

protein_metadata.head()


,accession,gene_symbol,organism,gene_id,isoform,sequence_length,ambiguous_x_count,stop_count,sequence,accession_priority,length_difference,length_ratio,length_status,passes_basic_qc
0,XP_083981534.1,atm,Abramis brama,311090226,NaN,3102,0,0,MSLALHELLLCCRGLENEKATERKKEVDRFRRLIRSPDTVEELDRT...,1,46,1.015052,Pass,True
1,XP_022052894.2,atm,Acanthochromis polyacanthus,110953284,X1,3092,0,0,MSLALNDLLLCCRGLESDKATERKKETERFRLLLQSPETIQELDRT...,1,36,1.011780,Pass,True
2,XP_036940904.1,atm,Acanthopagrus latus,119011683,NaN,3090,0,0,MSLALHDLMVCCRGLESDKATERKKVAEQFRRLIRTPETVQELDRS...,1,34,1.011126,Pass,True
3,XP_026892237.2,ATM,Acinonyx jubatus,106983963,X3,3057,0,0,MSLALNDLLICCRQLEHDRATERRKEVEKFKRLIRDPETVQHLDRH...,1,1,1.000327,Pass,True
4,XP_084640426.1,ATM,Acomys minous,311598838,1,3054,0,0,MSLALNDLLICCRQLEHDRATERRKEVEKFKRLIQDPETVQNLDRH...,1,2,0.999346,Pass,True


In [30]:
from Bio import SeqIO

# get protein fasta
fasta_file = (
    project_root
    / "data"
    / "processed"
    / "atm_representative_proteins.fasta"
)

protein_records = list(SeqIO.parse(fasta_file, "fasta"))

print(f"Protein loaded ({len(protein_records)})")
print(f"First protein ID ({protein_records[0].id})")
print(f"First protein description ({protein_records[0].description})")


Protein loaded (827)
First protein ID (XP_083981534.1)
First protein description (XP_083981534.1 ATM [organism=Abramis brama] [GeneID=311090226] [isoform=<NA>])


In [31]:
human_records = [record for record in protein_records if record.id == "NP_000042.3"]

sequence_lengths = pd.Series(
    [len(record) for record in protein_records]
)

print("Metadata rows:", len(protein_metadata))
print("FASTA records:", len(protein_records))
print("Unique FASTA IDs:", len({record.id for record in protein_records}))
print("Human reference records:", len(human_records))

print("\nSequence length summary:")
print(sequence_lengths.describe())

Metadata rows: 827
FASTA records: 827
Unique FASTA IDs: 827
Human reference records: 1

Sequence length summary:
count     827.00000
mean     3063.67110
std        49.54161
min      2331.00000
25%      3056.00000
50%      3064.00000
75%      3085.00000
max      3189.00000
dtype: float64


In [34]:
import subprocess

aligned_fasta = (
    project_root
    / "data"
    / "processed"
    / "atm_representative_proteins_aligned.fasta"
)

# Remove the empty file created by the failed attempt
if aligned_fasta.exists():
    aligned_fasta.unlink()

with aligned_fasta.open("w") as output_handle:
    alignment_run = subprocess.run(
        [
            "cmd.exe",
            "/C",
            str(mafft_path),
            "--auto",
            "--thread",
            "-1",
            str(fasta_file),
        ],
        stdout=output_handle,
        stderr=subprocess.PIPE,
        text=True,
    )

print("Return code:", alignment_run.returncode)
print("Output file exists:", aligned_fasta.exists())

if aligned_fasta.exists():
    print("Output file size:", aligned_fasta.stat().st_size, "bytes")

if alignment_run.returncode != 0:
    print(alignment_run.stderr)
    raise RuntimeError("MAFFT alignment failed.")

Return code: 0
Output file exists: True
Output file size: 3745802 bytes


Return code: 1
Alignment saved: True
Output file: c:\Users\Dell - i5 11th Gen\Desktop\atm-protein-conservation-explorer\data\processed\atm_representative_proteins_aligned.fasta
'\"C:\wamp64\Tools\mafft-7.526-win64-signed\mafft-win\mafft.bat\"' is not recognized as an internal or external command,
operable program or batch file.



RuntimeError: MAFFT alignment failed.